In [0]:
df = spark.table("cpg_planning.bronze.demand_statcan_retail")



In [0]:
# Shape of the data
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
print(f"Date range:")
display(df.selectExpr("min(ref_date)", "max(ref_date)"))


In [0]:

# How many rows per NAICS code
print("Rows per NAICS code:")
display(df.groupBy("naics_code", "naics_description")
    .count()
    .orderBy("naics_code"))



In [0]:
# How many suppressed values (status = 'x')
print("Suppressed values:")
display(df.groupBy("status").count())

In [0]:
display(df.groupBy("geo").count().orderBy("geo"))


In [0]:
# Check suppressed values by province
# Some smaller provinces may be heavily suppressed
from pyspark.sql.functions import col

display(df.filter(col("status") == "x")
    .groupBy("geo")
    .count()
    .orderBy("count", ascending=False))

In [0]:
# notebooks/01_demand_sensing/02_bronze_to_silver.py

from pyspark.sql.functions import col, sum as spark_sum

TARGET_NAICS = ["445", "456", "457", "458", "455"]

MAJOR_PROVINCES = [
    "Ontario", "Quebec", "British Columbia",
    "Alberta", "Manitoba", "Saskatchewan"
]

CLEAN_STATUSES = ["A", "B", "C", "D", "E", None]

df = spark.table("cpg_planning.bronze.demand_statcan_retail")

silver = (df
    .filter(col("naics_code").isin(TARGET_NAICS))
    .filter(col("geo").isin(MAJOR_PROVINCES))
    .filter(col("status").isin(CLEAN_STATUSES) | col("status").isNull())
    .select("ref_date", "geo", "naics_code", "naics_description", "value", "status"))

# Validate before writing
print(f"Silver rows: {silver.count()}")
display(silver.groupBy("naics_code", "geo").count().orderBy("naics_code", "geo"))

In [0]:
   (silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cpg_planning.silver.demand_retail_monthly"))

# Validate
count = spark.table("cpg_planning.silver.demand_retail_monthly").count()
print(f"Silver table rows: {count}")

In [0]:
# notebooks/01_demand_sensing/04_silver_to_gold.py
# Builds gold feature table from silver.
# All features lagged to prevent leakage — when predicting month T,
# only data available through month T-1 is used.

from pyspark.sql import Window
from pyspark.sql.functions import (
    col, lag, avg, round as spark_round
)

df = spark.table("cpg_planning.silver.demand_retail_monthly")

# Window partitioned by category + province, ordered by time
w = Window.partitionBy("naics_code", "geo").orderBy("ref_date")

features = (df
    .withColumn("lag_1m",  lag("value", 1).over(w))
    .withColumn("lag_2m",  lag("value", 2).over(w))
    .withColumn("lag_3m",  lag("value", 3).over(w))
    .withColumn("lag_6m",  lag("value", 6).over(w))
    .withColumn("lag_12m", lag("value", 12).over(w))
    .withColumn("rolling_3m_avg",
        spark_round(avg("value").over(w.rowsBetween(-3, -1)), 2))
    .withColumn("rolling_6m_avg",
        spark_round(avg("value").over(w.rowsBetween(-6, -1)), 2))
)

# Drop rows where lags are null (first 12 months per partition)
features = features.dropna(subset=["lag_12m"])

print(f"Feature rows: {features.count()}")
display(features.limit(5))

In [0]:
 # Spot check — Ontario food retail in order
# lag_1m should equal the previous row's value
display(features
    .filter((col("naics_code") == "445") & (col("geo") == "Ontario"))
    .select("ref_date", "value", "lag_1m", "lag_2m", "lag_12m", "rolling_3m_avg")
    .orderBy("ref_date")
    .limit(10))

In [0]:
from pyspark.sql.functions import col, abs as spark_abs, avg

df = spark.table("cpg_planning.gold.demand_feature_table")

display(df
    .withColumn("mape_last_month", 
        spark_abs((col("value") - col("lag_1m")) / col("value")))
    .withColumn("mape_same_month_ly", 
        spark_abs((col("value") - col("lag_12m")) / col("value")))
    .groupBy("naics_code", "naics_description")
    .agg(
        avg("mape_last_month").alias("mape_last_month"),
        avg("mape_same_month_ly").alias("mape_same_month_ly")
    )
    .orderBy("naics_code"))

In [0]:
import mlflow

category_mapes = df \
    .withColumn("mape_last_month",
        spark_abs((col("value") - col("lag_1m")) / col("value"))) \
    .withColumn("mape_same_month_ly",
        spark_abs((col("value") - col("lag_12m")) / col("value"))) \
    .groupBy("naics_code") \
    .agg(
        avg("mape_last_month").alias("mape_last_month"),
        avg("mape_same_month_ly").alias("mape_same_month_ly")
    ).collect()

with mlflow.start_run(run_name="baseline_per_category"):
    for row in category_mapes:
        mlflow.log_metric(f"{row['naics_code']}_mape_last_month", 
                         row["mape_last_month"])
        mlflow.log_metric(f"{row['naics_code']}_mape_same_month_ly", 
                         row["mape_same_month_ly"])
    print("Per-category baselines logged to MLflow")